In [1]:
# -*- coding: utf-8 -*-
"""
Created on Wed Mar  4 20:40:09 2026

@author: THE MAD TITAN
"""

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Conv2D
import numpy as np
import logging
tf.get_logger().setLevel(logging.ERROR)

EPOCHS = 128
BATCH_SIZE = 512

# Load dataset.
cifar_dataset = keras.datasets.cifar10
(train_images, train_labels), (test_images,
    test_labels) = cifar_dataset.load_data()

# Standardize dataset.
mean = np.mean(train_images)
stddev = np.std(train_images)
train_images = (train_images - mean) / stddev
test_images = (test_images - mean) / stddev

print('mean: ', mean)
print('stddev: ', stddev)

# Change labels to one-hot.
train_labels = to_categorical(train_labels,
                              num_classes=10)
test_labels = to_categorical(test_labels,
                              num_classes=10)


# making AlexNet.
model = Sequential()
model.add(Conv2D(96, (11, 11), strides=(4,4),
                activation='relu', padding='same',
                input_shape=(32, 32, 3),
                kernel_initializer='he_normal',
                bias_initializer='zeros'))

model.add(MaxPooling2D(pool_size=(3, 3), strides=(2,2), padding='same'))

model.add(Conv2D(256, (5, 5), strides=(1,1),
                activation='relu', padding='same',
                kernel_initializer='he_normal',
                bias_initializer='zeros'))

# adding maxpooling
model.add(MaxPooling2D(pool_size=(3, 3), strides=(2,2), padding='same'))

model.add(Conv2D(384, (3, 3), strides=(1,1),
                activation='relu', padding='same',
                kernel_initializer='he_normal',
                bias_initializer='zeros'))
model.add(Conv2D(384, (3, 3), strides=(1,1),
                activation='relu', padding='same',
                kernel_initializer='he_normal',
                bias_initializer='zeros'))
model.add(Conv2D(256, (3, 3), strides=(1,1),
                activation='relu', padding='same',
                kernel_initializer='he_normal',
                bias_initializer='zeros'))

# adding maxpooling
model.add(MaxPooling2D(pool_size=(3, 3), strides=(2,2), padding='same'))

model.add(Flatten())
model.add(Dense(4096, activation='relu',
                kernel_initializer='glorot_uniform',
                bias_initializer='zeros'))
model.add(Dense(4096, activation='relu',
                kernel_initializer='glorot_uniform',
                bias_initializer='zeros'))
model.add(Dense(10, activation='softmax',
                kernel_initializer='glorot_uniform',
                bias_initializer='zeros'))

model.compile(loss='categorical_crossentropy',
              optimizer='adam', metrics =['accuracy'])
model.summary()
history = model.fit(
    train_images, train_labels, validation_data =
    (test_images, test_labels), epochs=EPOCHS,
    batch_size=BATCH_SIZE, verbose=2, shuffle=True)



import matplotlib.pyplot as plt

# ============================================================
# 1. TRAINING HISTORY - Train Error vs Test Error (Slide 6)
# ============================================================
train_error = [1 - acc for acc in history.history['accuracy']]
test_error  = [1 - acc for acc in history.history['val_accuracy']]

plt.figure(figsize=(8, 5))
plt.plot(train_error, color='red',  label='train error')
plt.plot(test_error,  color='green', label='test error')
plt.xlabel('training epochs')
plt.ylabel('error')
plt.title('Learning for CIFAR-10')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 8. LIVE TRAINING LOSS & ACCURACY PER EPOCH (Bonus)
# ============================================================
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history['loss'],     label='Train Loss', color='red')
plt.plot(epochs_range, history.history['val_loss'], label='Test Loss',  color='green')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss per Epoch')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history['accuracy'],     label='Train Accuracy', color='red')
plt.plot(epochs_range, history.history['val_accuracy'], label='Test Accuracy',  color='green')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy per Epoch')
plt.legend()
plt.grid(True)

plt.suptitle('Your Model Training Progress')
plt.tight_layout()
plt.show()

# ============================================================
# 9. SAMPLE PREDICTIONS - What is the model seeing? (Bonus)
# ============================================================
class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

# Get raw test images (before standardization) for display
cifar_dataset = tf.keras.datasets.cifar10
(_, _), (raw_test_images, raw_test_labels) = cifar_dataset.load_data()

predictions = model.predict(test_images[:16])
pred_classes = np.argmax(predictions, axis=1)
true_classes = raw_test_labels[:16].flatten()

plt.figure(figsize=(12, 6))
for i in range(16):
    plt.subplot(2, 8, i+1)
    plt.imshow(raw_test_images[i])
    color = 'green' if pred_classes[i] == true_classes[i] else 'red'
    plt.title(f'P:{class_names[pred_classes[i]]}\nT:{class_names[true_classes[i]]}',
              fontsize=6, color=color)
    plt.axis('off')

plt.suptitle('Model Predictions (Green=Correct, Red=Wrong)')
plt.tight_layout()
plt.show()

  3956736/170498071 ━━━━━━━━━━━━━━━━━━━━ 33:35 12us/step

KeyboardInterrupt: 